In [ ]:
# in A100 서버

In [1]:
import os
import warnings
import logging
import torch

# Optional imports (only needed if you run CLIP/CLAP feature extraction later)
try:
    import clip
except Exception as e:
    clip = None
    print(f"[WARN] clip import failed: {e}")

try:
    import laion_clap
except Exception as e:
    laion_clap = None
    print(f"[WARN] laion_clap import failed: {e}")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger('librosa').setLevel(logging.ERROR)


/home/user/anaconda3/envs/qx_dllm/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


PyTorch version: 2.7.1+cu118
CUDA available: True


In [2]:
import os
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple

# =========================
# 1) KETI path config
# =========================
KETI_BASE = "/ssd_data/qx/KETIdata_by_scenario/speaking_only/divided_by_person/SCENE3"
TRAIN_DIR = os.path.join(KETI_BASE, "train30s")
VAL_DIR   = os.path.join(KETI_BASE, "valid30s")

LABEL_CSV = "/ssd_data/qx/KETIdata/keti_corpus_labels.csv"

# 你当前使用的 5 个标签键（固定）
FEATURES = ['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']


# =========================
# 2) build dictionary: {ID(int): {openness:..., ...}}
# =========================
def build_keti_dictionary(csv_path: str, encoding: str = "utf-8-sig") -> Dict[int, Dict[str, float]]:
    df = pd.read_csv(csv_path, encoding=encoding)

    # 自动猜一下ID列名；如果你确定ID列名，就直接写死更稳
    id_candidates = ["ID", "id", "participant", "participant_id", "subject", "subject_id", "person_id", "PersonID"]
    id_col = next((c for c in id_candidates if c in df.columns), None)
    if id_col is None:
        raise ValueError(f"找不到ID列。你csv列名是: {list(df.columns)}")

    # 确保 FEATURES 都存在
    miss = [c for c in FEATURES if c not in df.columns]
    if miss:
        raise ValueError(f"CSV缺少这些标签列: {miss}\n当前列名: {list(df.columns)}")

    df = df[[id_col] + FEATURES].copy()
    df[id_col] = df[id_col].astype(str).str.extract(r"(\d+)")[0].astype(int)
    for c in FEATURES:
        df[c] = df[c].astype(float)

    dictionary = df.set_index(id_col)[FEATURES].to_dict(orient="index")
    return dictionary


# =========================
# 3) your label reading logic (KETI filename rule: first 3 chars)
# =========================
def extract_third_character(filename: str) -> str:
    # filename can be full path; we only want basename
    filename = os.path.basename(filename)
    return filename[:3]

def reading_label_data(file_name: str, dictionary: Dict[int, Dict[str, float]]) -> np.ndarray:
    index = int(extract_third_character(file_name))  # "001" -> 1
    gt = dictionary.get(index, None)
    if gt is None:
        raise KeyError(f"ID not found in dictionary: {index} (from file {file_name})")

    extracted_data = [float(gt[label]) for label in FEATURES]
    return np.stack(extracted_data).reshape(5, 1)  # (5,1)


# =========================
# 4) scan split dir and build (paths, labels)
# =========================
def load_keti_split(
    split_dir: str,
    dictionary: Dict[int, Dict[str, float]],
    exts=(".mp4", ".avi", ".mov", ".mkv"),
    recursive: bool = True,
    strict: bool = False,   # True: 遇到无label/异常文件就报错；False: 跳过
) -> Tuple[List[str], np.ndarray]:
    video_paths: List[str] = []
    label_list: List[np.ndarray] = []

    walker = os.walk(split_dir) if recursive else [(split_dir, [], os.listdir(split_dir))]

    for root, _, files in walker:
        for fn in files:
            if not fn.lower().endswith(exts):
                continue
            fpath = os.path.join(root, fn)

            # 文件名必须以3位数字开头
            base = os.path.basename(fn)
            if len(base) < 3 or not base[:3].isdigit():
                if strict:
                    raise ValueError(f"Bad filename (need first 3 digits): {base}")
                continue

            try:
                y = reading_label_data(base, dictionary)   # (5,1)
            except Exception as e:
                if strict:
                    raise
                else:
                    continue

            video_paths.append(fpath)
            label_list.append(y)

    # (N, 5) 更常用；如果你后面一定要 (N,5,1) 也可以不 squeeze
    labels = np.stack(label_list, axis=0).astype(np.float32)  # (N,5,1)
    # labels = labels.squeeze(-1)  # -> (N,5) 需要的话取消注释
    return video_paths, labels


# =========================
# 5) run
# =========================
dictionary = build_keti_dictionary(LABEL_CSV)

train_v_paths, train_labels = load_keti_split(TRAIN_DIR, dictionary)
val_v_paths,   val_labels   = load_keti_split(VAL_DIR, dictionary)

print("Train:", len(train_v_paths), train_labels.shape)  # (N,5,1)
print("Val:  ", len(val_v_paths),   val_labels.shape)

print("Example:", os.path.basename(train_v_paths[0]), "->", train_labels[0].reshape(-1))


Train: 4122 (4122, 5, 1)
Val:   1346 (1346, 5, 1)
Example: 148_D016_S03_L4_SCENE3_005.mp4 -> [91. 69. 70. 76. 77.]


In [3]:
import os
import re
import cv2
import torch
import librosa
import numpy as np
import h5py
import pandas as pd
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset


# -----------------------------
# helpers for clip naming
# -----------------------------
_CLIPIDX_RE = re.compile(r"_(\d{3})\.(mp4|avi|mov|mkv)$", re.IGNORECASE)

def parse_clip_index(filename: str):
    """001_D001_S01_L1_SCENE3_003.mp4 -> 3"""
    m = _CLIPIDX_RE.search(os.path.basename(filename))
    return int(m.group(1)) if m else None

def parse_base_id_from_clip(filename: str):
    """001_D001_S01_L1_SCENE3_003.mp4 -> 001_D001_S01_L1_SCENE3"""
    filename = os.path.basename(filename)
    m = _CLIPIDX_RE.search(filename)
    if not m:
        return os.path.splitext(filename)[0]
    return filename[:m.start()]  # remove _003.ext

def parse_scene_from_clip(filename: str):
    """robust: split by '_' and take the token like SCENE3"""
    parts = os.path.basename(filename).split("_")
    for p in parts:
        if p.startswith("SCENE"):
            return p
    return os.path.basename(filename)[16:22]


# -----------------------------
# Assign whole sentence to ONE clip by max overlap (tie -> earlier clip)
# -----------------------------
def assign_whole_sentence_to_one_clip(sent_start: float, sent_end: float, clip_len: float = 30.0):
    """
    Return: (clip_idx_1based, best_overlap_seconds)

    Rule:
      - a sentence that crosses clip boundaries is assigned wholly to the clip with max overlap
      - if tie, assign to earlier clip
    """
    if sent_end <= sent_start:
        k = int(sent_start // clip_len) + 1
        return k, 0.0

    k0 = int(sent_start // clip_len)
    k1 = int((sent_end - 1e-9) // clip_len)

    best_k = k0
    best_ov = -1.0
    for k in range(k0, k1 + 1):
        bin_s = k * clip_len
        bin_e = (k + 1) * clip_len
        ov = max(0.0, min(sent_end, bin_e) - max(sent_start, bin_s))
        if (ov > best_ov) or (abs(ov - best_ov) < 1e-9 and k < best_k):
            best_ov = ov
            best_k = k

    return best_k + 1, best_ov


# -----------------------------
# Build clip->transcription map from sentence-level CSV
#   - robust numeric parsing
#   - skip bad lines
#   - whole sentence goes to ONE clip by max overlap
#   - print every video's transcription
# -----------------------------
def build_clip_transcription_map(
    sentence_csv: str,
    video_paths,
    encoding="utf-8",
    clip_len: float = 30.0,
    print_each: bool = True,
):
    """
    sentence_csv columns expected:
      id,count,start,end,script,SCENE,Folder,SourceFile
    We will:
      1) build speech-only timeline by accumulating each sentence duration in order
      2) map each sentence to ONE 30s clip by max overlap
      3) for each clip video file, concatenate its assigned sentences
    """
    df = pd.read_csv(
        sentence_csv,
        encoding=encoding,
        dtype={"Folder": str},
        on_bad_lines="skip"   # avoid crash on malformed rows
    )

    required = ["start", "end", "script", "SCENE", "Folder", "SourceFile"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Sentence CSV missing columns: {missing}\n"
            f"Current columns: {list(df.columns)}"
        )

    # robust numeric conversion (bad -> NaN -> drop)
    df["start"] = pd.to_numeric(df["start"], errors="coerce")
    df["end"]   = pd.to_numeric(df["end"], errors="coerce")
    bad = df[df["start"].isna() | df["end"].isna()]
    if len(bad) > 0:
        print(f"[WARN] Dropping bad rows where start/end not numeric: {len(bad)}")
        print(bad.head(3)[["id", "start", "end", "script", "SourceFile"]])

    df = df.dropna(subset=["start", "end"]).copy()
    df["script"] = df["script"].fillna("").astype(str)
    df["SCENE"]  = df["SCENE"].astype(str)
    df["Folder"] = df["Folder"].astype(str).str.zfill(3)
    df["SourceFile"] = df["SourceFile"].astype(str)
    df["base_id"] = df["SourceFile"].str.replace(".csv", "", regex=False)

    # filter to base_ids used by current video_paths
    used_base_ids = {parse_base_id_from_clip(os.path.basename(vp)) for vp in video_paths}
    df = df[df["base_id"].isin(used_base_ids)].copy()
    if len(df) == 0:
        print("[WARN] base_id mismatch: no rows matched your video_paths. All transcriptions will be empty.")
        clip_trans_map = {os.path.basename(vp): "" for vp in video_paths}
        if print_each:
            for bn in sorted(clip_trans_map.keys()):
                print(f"{bn}\t{clip_trans_map[bn]}")
        return clip_trans_map

    # (Folder, SCENE, base_id, clip_idx) -> list of (speech_start, script)
    bucket = {}

    for (folder, scene, base_id), g in df.groupby(["Folder", "SCENE", "base_id"]):
        g = g.sort_values("start").reset_index(drop=True)

        speech_t = 0.0  # speech-only timeline
        for _, row in g.iterrows():
            dur = float(row["end"]) - float(row["start"])
            if dur <= 0:
                continue

            sent_s = speech_t
            sent_e = speech_t + dur
            speech_t = sent_e

            clip_idx, _ = assign_whole_sentence_to_one_clip(sent_s, sent_e, clip_len=clip_len)
            key = (folder, scene, base_id, clip_idx)
            bucket.setdefault(key, []).append((sent_s, row["script"]))

    # build per-video transcription
    clip_trans_map = {}
    for vp in video_paths:
        bn = os.path.basename(vp)
        folder = bn[:3]
        scene = parse_scene_from_clip(bn)
        base_id = parse_base_id_from_clip(bn)
        clip_idx = parse_clip_index(bn)

        if clip_idx is None:
            clip_trans_map[bn] = ""
            continue

        key = (folder, scene, base_id, clip_idx)
        items = bucket.get(key, [])
        if not items:
            clip_trans_map[bn] = ""
        else:
            items.sort(key=lambda x: x[0])
            clip_trans_map[bn] = " ".join([s.strip() for _, s in items if s.strip()])

    if print_each:
        print("\n[INFO] Per-video transcription:")
        for vp in video_paths:
            bn = os.path.basename(vp)
            txt = clip_trans_map.get(bn, "")
            print(f"{bn}\t{txt}")

    return clip_trans_map


# =============================
# Dataset
# =============================
class PersonalityDataset_Full(Dataset):
    """
    KETI SCENE3 -> H5 builder

    - video: 16 frames
    - audio: librosa from video
    - label: Big5
    - transcription: from sentence CSV, whole sentence assigned to one 30s clip by max overlap
    """
    def __init__(
        self,
        video_paths,
        labels,
        clip_preprocess,
        sentence_transcriptions_csv,
        sampling_rate=48000,
        duration=7,
        sentence_csv_encoding="utf-8",
        clip_len=30.0,
        print_each_transcription=True,
    ):
        self.video_paths = video_paths
        self.labels = labels
        self.clip_preprocess = clip_preprocess
        self.sr = sampling_rate
        self.duration = duration
        self.max_audio_samples = sampling_rate * duration

        # build transcription mapping ONCE and print each video's transcription
        self.clip_trans_map = build_clip_transcription_map(
            sentence_csv=sentence_transcriptions_csv,
            video_paths=self.video_paths,
            encoding=sentence_csv_encoding,
            clip_len=clip_len,
            print_each=print_each_transcription,
        )

    def __len__(self):
        return len(self.video_paths)

    def _get_video_frames(self, path, num_frames=16):
        cap = cv2.VideoCapture(path)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if frame_count <= 0:
            cap.release()
            return torch.zeros((num_frames, 3, 224, 224), dtype=torch.float32)

        indices = np.linspace(0, frame_count - 1, num_frames, dtype=int)
        frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ret, frame = cap.read()
            if not ret or frame is None:
                frame = np.zeros((224, 224, 3), dtype=np.uint8)
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(self.clip_preprocess(Image.fromarray(frame)))
        cap.release()
        return torch.stack(frames)

    def _get_audio_waveform(self, video_path):
        try:
            audio, _ = librosa.load(video_path, sr=self.sr, mono=True, duration=self.duration)
            if len(audio) > self.max_audio_samples:
                audio = audio[:self.max_audio_samples]
            else:
                audio = np.pad(audio, (0, self.max_audio_samples - len(audio)), mode='constant')
            return np.nan_to_num(audio).astype(np.float32)
        except Exception:
            return np.zeros(self.max_audio_samples, dtype=np.float32)

    def _get_transcription(self, video_path):
        bn = os.path.basename(video_path)
        return self.clip_trans_map.get(bn, "")


# =============================
# H5 保存（带清晰进度条）
# =============================
def save_full_dataset_to_h5(dataset_obj, save_path):
    num_samples = len(dataset_obj.video_paths)

    with h5py.File(save_path, 'w') as f:
        f.create_dataset('video', (num_samples, 16, 3, 224, 224), dtype='float32',
                         chunks=(1, 16, 3, 224, 224))
        f.create_dataset('audio', (num_samples, dataset_obj.max_audio_samples), dtype='float32',
                         chunks=(1, dataset_obj.max_audio_samples))
        f.create_dataset('label', (num_samples, 5), dtype='float32')

        dt = h5py.special_dtype(vlen=str)
        f.create_dataset('transcription', (num_samples,), dtype=dt)

        pbar = tqdm(total=num_samples, desc=f"Saving {os.path.basename(save_path)}", dynamic_ncols=True, leave=True)
        for i, path in enumerate(dataset_obj.video_paths):
            try:
                f['video'][i] = dataset_obj._get_video_frames(path).numpy()
                f['audio'][i] = dataset_obj._get_audio_waveform(path)
                f['label'][i] = dataset_obj.labels[i].reshape(-1)
                f['transcription'][i] = dataset_obj._get_transcription(path)
            except Exception as e:
                print(f"\nError at {path}: {e}")
            finally:
                pbar.update(1)
        pbar.close()


# =============================
# RUN
# =============================
import clip

device = "cuda" if torch.cuda.is_available() else "cpu"
_, preprocess = clip.load("ViT-L/14", device=device)

# ✅ 你的 sentence-level CSV（注意：路径里有空格的话要确保文件名真实存在）
transcriptions_filepath = "/ssd_data/qx/KETIdata/each_user_transcriptions cp.csv"

# 这里假设你已经有：
#   train_v_paths, train_labels
#   val_v_paths, val_labels

train_ds = PersonalityDataset_Full(
    train_v_paths,
    train_labels,
    preprocess,
    transcriptions_filepath,
    print_each_transcription=False,   # ✅ 打印每个视频的 transcription
)

val_ds = PersonalityDataset_Full(
    val_v_paths,
    val_labels,
    preprocess,
    transcriptions_filepath,
    print_each_transcription=False,   # ✅ 打印每个视频的 transcription
)

save_full_dataset_to_h5(train_ds, "keti_scene3_train.h5")
save_full_dataset_to_h5(val_ds,   "keti_scene3_val.h5")


/tmp/ipykernel_1562220/3528671783.py:94: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/tmp/ipykernel_1562220/3528671783.py:94: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
Saving keti_scene3_val.h5: 100%|██████████| 1346/1346 [05:24<00:00,  4.15it/s]


In [ ]:
import h5py
import numpy as np

def inspect_h5_file(file_path, num_samples=3):
    print(f"🧐 파일 확인 중: {file_path}")
    
    with h5py.File(file_path, 'r') as f:
        # 1. 사용 가능한 키(Datasets) 확인
        print(f"🔑 Keys in H5: {list(f.keys())}")
        
        # 2. 각 데이터셋의 Shape 확인
        for key in f.keys():
            print(f"📊 Dataset: {key} | Shape: {f[key].shape} | Dtype: {f[key].dtype}")
        
        print("-" * 50)
        
        # 3. 실제 샘플 데이터 출력
        for i in range(min(num_samples, len(f['label']))):
            print(f"--- Sample {i} ---")
            
            # 텍스트 데이터 (transcription)
            # h5py에서 읽어온 문자열은 bytes 타입일 수 있으므로 decode가 필요할 수 있습니다.
            transcription = f['transcription'][i]
            if isinstance(transcription, bytes):
                transcription = transcription.decode('utf-8')
            
            print(f"📝 [Transcription]: {transcription}")
            
            # 레이블 데이터 (Big Five Personality)
            print(f"🎯 [Labels]: {f['label'][i]}")
            
            # 비디오/오디오 데이터 요약
            print(f"📹 [Video Shape]: {f['video'][i].shape}")
            print(f"🔊 [Audio Max/Min]: {f['audio'][i].max():.4f} / {f['audio'][i].min():.4f}")
            print("-" * 50)

# 실행
inspect_h5_file("udiva_train_data_final.h5")

In [ ]:
import pickle
import os

def check_pkl_contents(file_path):
    print(f"🔍 파일 확인 중: {os.path.basename(file_path)}")
    
    try:
        with open(file_path, 'rb') as f:
            # FirstImpression 데이터셋 표준인 latin1 인코딩 사용
            data = pickle.load(f, encoding='latin1')
        
        # 1. 전체 데이터 타입 및 개수 확인
        print(f"📊 데이터 타입: {type(data)}")
        print(f"🔢 총 아이템 개수: {len(data)}")
        
        # 2. 첫 번째 샘플 데이터 확인
        if isinstance(data, dict):
            first_key = list(data.keys())[0]
            print(f"✅ Key 샘플 (형식 확인): '{first_key}'")
            print(f"📝 Value 샘플 (내용 확인): '{data[first_key]}'")
        elif isinstance(data, list):
            print(f"✅ 리스트 형태입니다. 첫 번째 요소: {data[0]}")
            
    except Exception as e:
        print(f"❌ 에러 발생: {e}")
    print("-" * 50)

# 이미지에 있는 파일 경로들을 입력하세요
# BASE_DIR 설정에 맞춰 경로를 수정해주세요.
check_pkl_contents("./Big5/transcription/transcription_training.pkl")
check_pkl_contents("./Big5/transcription/transcription_validation.pkl")

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import clip
import laion_clap
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm
from peft import LoraConfig, get_peft_model

# 직접 만든 모듈 import
from llama_model import PersonalityLLM
from h5personalityDataset import H5PersonalityDataset 

# ==========================================
# 1. 경로 및 하이퍼파라미터 설정
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
EPOCHS = 30
LR = 1e-6
TRAIN_H5 = "train_data_final.h5"
VAL_H5 = "val_data_final.h5"

# ==========================================
# 2. 모델 및 인코더 초기화
# ==========================================
print("🚀 모델 및 Frozen 인코더 로딩 중...")
clip_model, _ = clip.load("ViT-L/14", device=device)
clap_model = laion_clap.CLAP_Module(enable_fusion=False).to(device)
clap_model.load_ckpt() 

model = PersonalityLLM(device=device).to(device)

# 가중치 정밀도 설정 (Llama 3 bfloat16 호환)
model.video_projector.to(torch.bfloat16)
model.audio_projector.to(torch.bfloat16)
model.regression_head.to(torch.float32)

# LoRA 적용
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    modules_to_save=["embed_tokens"]  # <--- 이 부분을 추가하여 스페셜 토큰 임베딩이 학습되게 합니다.
)
model.llama = get_peft_model(model.llama, lora_config)
model.llama.print_trainable_parameters()

# ==========================================
# 3. 데이터셋 및 고속 로더 준비
# ==========================================
if not os.path.exists(TRAIN_H5) or not os.path.exists(VAL_H5):
    raise FileNotFoundError("H5 파일이 존재하지 않습니다. 전처리 스크립트를 먼저 실행하세요.")

train_dataset = H5PersonalityDataset(TRAIN_H5)
val_dataset = H5PersonalityDataset(VAL_H5)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

# ==========================================
# 4. 최적화 도구 및 기록용 리스트
# ==========================================
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.L1Loss() 

def get_accuracy(pred, target):
    return 1 - torch.abs(pred - target).mean()

# 결과 기록용 딕셔너리
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}
best_acc = -1e9

# ==========================================
# 5. 메인 학습 루프
# ==========================================
for epoch in range(EPOCHS):
    # --- TRAIN ---
    model.train()
    train_loss_sum, train_acc_sum = 0.0, 0.0
    pbar = tqdm(train_loader, desc=f"🔥 [TRAIN] Ep {epoch+1}/{EPOCHS}")
    
    for batch in pbar:
        videos = batch["video"].to(device)           # (B, 16, 3, 224, 224)
        audios = batch["audio"].to(device).float()   # (B, Samples)
        target_labels = batch["label"].to(device)
        transcriptions = batch["transcription"]
        b = videos.shape[0]

        with torch.no_grad():
            # CLIP 특징 추출
            v_feat = clip_model.encode_image(videos.view(-1, 3, 224, 224))
            v_feat = v_feat.view(b, 16, -1).mean(dim=1).to(torch.bfloat16)
            # CLAP 특징 추출
            a_feat = clap_model.get_audio_embedding_from_data(x=audios, use_tensor=True)
            if isinstance(a_feat, np.ndarray): a_feat = torch.from_numpy(a_feat).to(device)
            a_feat = a_feat.to(torch.bfloat16)

        prompts = [
                    f"The person said: '{t}'. Based on this speech, video, and audio, determine the Big5 personality scores." 
                    if t.strip() else "Determine the Big5 personality scores based on the video and audio."
                    for t in transcriptions
                ]        
        
        inputs = model.tokenizer(
            prompts, 
            return_tensors="pt", 
            padding=True, 
            truncation=True, 
            max_length=256 # 적절한 길이 제한
        ).to(device)

        preds = model(
            input_ids=inputs.input_ids, 
            video_features=v_feat, 
            audio_features=a_feat,
            attention_mask=inputs.attention_mask # 마스크 전달 필수
        )
        loss = criterion(preds, target_labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        acc = get_accuracy(preds.detach(), target_labels)
        train_loss_sum += loss.item()
        train_acc_sum += acc.item()
        pbar.set_postfix({"Loss": f"{loss.item():.4f}", "Acc": f"{acc.item():.4f}"})

    # Train 결과 저장
    history['train_loss'].append(train_loss_sum / len(train_loader))
    history['train_acc'].append(train_acc_sum / len(train_loader))

    # --- VALIDATION ---
    model.eval()
    val_loss_sum, val_acc_sum = 0.0, 0.0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="🛡️ [VAL]"):
            videos = batch["video"].to(device)
            audios = batch["audio"].to(device).float()
            target_labels = batch["label"].to(device)
            transcriptions = batch["transcription"]
            b = videos.shape[0]

            v_feat = clip_model.encode_image(videos.view(-1, 3, 224, 224)).view(b, 16, -1).mean(dim=1).to(torch.bfloat16)
            a_feat = clap_model.get_audio_embedding_from_data(x=audios, use_tensor=True)
            if isinstance(a_feat, np.ndarray): a_feat = torch.from_numpy(a_feat).to(device)
            a_feat = a_feat.to(torch.bfloat16)

            prompts = [
                f"The person said: '{t}'. Based on this speech, video, and audio, determine the Big5 personality scores." 
                if t.strip() else "Determine the Big5 personality scores based on the video and audio."
                for t in transcriptions
            ]
            inputs = model.tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
            
            preds = model(inputs.input_ids, v_feat, a_feat, attention_mask=inputs.attention_mask)
            
            val_loss_sum += criterion(preds, target_labels).item()
            val_acc_sum += get_accuracy(preds, target_labels).item()

    # Val 결과 저장
    avg_val_loss = val_loss_sum / len(val_loader)
    avg_val_acc = val_acc_sum / len(val_loader)
    history['val_loss'].append(avg_val_loss)
    history['val_acc'].append(avg_val_acc)

    print(f"📊 Result Ep {epoch+1} | Val Loss: {avg_val_loss:.4f} | Val 1-MAE: {avg_val_acc:.4f}")

    if avg_val_acc > best_acc:
        best_acc = avg_val_acc
        torch.save(model.state_dict(), "best_personality_model.pt")
        print("🏆 New Best Model Saved!\n")

# ==========================================
# 6. 학습 결과 시각화
# ==========================================
print("\n📈 학습 결과 그래프 생성 중...")
epochs_range = range(1, EPOCHS + 1)
plt.figure(figsize=(12, 5))

# Loss 그래프
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_loss'], label='Train Loss', color='#1f77b4')
plt.plot(epochs_range, history['val_loss'], label='Val Loss', color='#d62728', linestyle='--')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (L1)')
plt.legend()
plt.grid(True, alpha=0.3)

# Accuracy 그래프
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['train_acc'], label='Train 1-MAE', color='#2ca02c')
plt.plot(epochs_range, history['val_acc'], label='Val 1-MAE', color='#ff7f0e', linestyle='--')
plt.title('Training and Validation Accuracy (1-MAE)')
plt.xlabel('Epochs')
plt.ylabel('Accuracy Score')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('learning_results.png')
plt.show()

print("✅ 모든 프로세스가 완료되었습니다. 그래프: learning_results.png")
